# MOLM-ST Fixed-Budget Pareto Evaluation

This notebook evaluates the **original MOLM-ST** control under the fixed-budget multi-objective prioritization protocol used for the extended analysis.

### Automatic Kaggle input detection

The notebook can reuse:
1. the ESM representation cache; and
2. a compatible `MOLM_Unified_Experiment_Results.zip` bundle when available.

### Experiment

MOLM-ST is trained independently for the target-binding and OVA-binding proxy tasks using the same focal + ranking + gap task objective used by Standard-MOLM. Candidate prioritization is then evaluated at identical budgets `K={5,10,15,20,25}` using Recall, Precision, Enrichment, hypervolume, and IGD.


In [ ]:

from pathlib import Path
import os, sys, json, time, shutil, subprocess, platform, hashlib, math
import numpy as np
import pandas as pd

# ----------------------------
# Reproducibility configuration
# ----------------------------
REPO_URL = "https://github.com/DigantaX/molm-pipeline.git"
PINNED_COMMIT = "c5923984f0d5176977edb4a4ffd8fc5f98536043"

SEEDS = [42, 123, 456, 789, 2024]
FEATURE_TYPES = ["onehot", "mean_esm2", "mean_fusion", "site_esm2", "site_fusion"]
FEATURE_DIMS = {
    "onehot": 2300,
    "mean_esm2": 320,
    "mean_fusion": 2620,
    "site_esm2": 2560,
    "site_fusion": 4860,
}
K_VALUES = [5, 10, 15, 20, 25]
FULL_CURVE_MAX_K = 25

EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 5e-5

# Keep True for the complete cross-model comparison.
# Set False only if you intentionally want the original-paper 3-feature subset.
RUN_SITE_FEATURES = True
if not RUN_SITE_FEATURES:
    FEATURE_TYPES = ["onehot", "mean_esm2", "mean_fusion"]

REQUIRE_T4_X2 = True
RESUME = True
AUTO_DETECT_ESM_REUSE = True
AUTO_EXTRACT_RESULTS_ZIP = True
# NPY files have no sequence IDs; we accept them only after strict row-count/dimension checks.
# This is appropriate for the previously generated esm_reuse dataset from the same sequence ordering.
ALLOW_SHAPE_VERIFIED_NPY_REUSE = True

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/molm_st_fixed_budget")
REPO = WORK_ROOT / "molm-pipeline"
FEATURE_DIR = WORK_ROOT / "features"
JOBS_DIR = WORK_ROOT / "jobs"
LOG_DIR = WORK_ROOT / "logs"
ANALYSIS_DIR = WORK_ROOT / "analysis"
CODE_DIR = WORK_ROOT / "code"
EXISTING_RESULTS_DIR = WORK_ROOT / "existing_results"
REUSE_STAGE_DIR = WORK_ROOT / "reuse_stage"

for d in [WORK_ROOT, FEATURE_DIR, JOBS_DIR, LOG_DIR, ANALYSIS_DIR, CODE_DIR, EXISTING_RESULTS_DIR, REUSE_STAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Features:", FEATURE_TYPES)
print("Seeds:", SEEDS)



## 1. T4×2 preflight, dependencies, and pinned repository

For reproducibility this checks out the same repository commit used by the extended-analysis experiment bundle.


In [ ]:

import importlib.metadata
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPUs:", [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())])

if REQUIRE_T4_X2 and torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Please select Kaggle accelerator 'GPU T4 x2'. Found {torch.cuda.device_count()} GPU(s)."
    )

# fair-esm 2.0.0 is the version used by the definitive feature builder.
try:
    esm_version = importlib.metadata.version("fair-esm")
except importlib.metadata.PackageNotFoundError:
    esm_version = None

if esm_version != "2.0.0":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fair-esm==2.0.0"],
        check=True
    )
print("fair-esm:", importlib.metadata.version("fair-esm"))

# Reuse a mounted repository snapshot if one is supplied as a Kaggle dataset.
repo_candidates = []
if INPUT_ROOT.exists():
    for p in INPUT_ROOT.rglob("phase0_config.py"):
        parent = p.parent
        if (parent / "data" / "emi_binding.csv").exists():
            repo_candidates.append(parent)

if REPO.exists() and not RESUME:
    shutil.rmtree(REPO)

if not REPO.exists():
    if repo_candidates:
        shutil.copytree(repo_candidates[0], REPO)
        print("Using repository snapshot from Kaggle input:", repo_candidates[0])
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)

# Always force the exact commit.
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, (commit, PINNED_COMMIT)

print("Pinned repository ready:", REPO)
print("Commit:", commit)


## 1B. Automatically detect the ESM cache and unified result bundle

The notebook recursively searches attached Kaggle inputs for reusable ESM features and a compatible `MOLM_Unified_Experiment_Results.zip`. The result archive is extracted only once and an extraction marker is written for resumability.


In [ ]:

from pathlib import Path
import zipfile, shutil, os, json, re

def recursive_find(root: Path, predicate):
    if not root.exists():
        return []
    out = []
    for p in root.rglob("*"):
        try:
            if predicate(p):
                out.append(p)
        except Exception:
            pass
    return sorted(out)

# ----------------------------
# Detect ESM reuse dataset root
# ----------------------------
ESM_REUSE_ROOTS = []
if AUTO_DETECT_ESM_REUSE:
    # First prefer paths whose parent hierarchy explicitly contains "esm_reuse".
    explicit = recursive_find(
        INPUT_ROOT,
        lambda p: p.is_dir() and "esm_reuse" in str(p).lower()
    )
    # Keep only shallowest matching directories so we do not scan the same tree repeatedly.
    explicit = sorted(explicit, key=lambda p: len(p.parts))
    pruned = []
    for p in explicit:
        if not any(parent in p.parents or parent == p for parent in pruned):
            pruned.append(p)
    ESM_REUSE_ROOTS = pruned

print("Detected esm_reuse roots:")
if ESM_REUSE_ROOTS:
    for p in ESM_REUSE_ROOTS:
        print("  ", p)
else:
    print("   none")

# ----------------------------------------
# Detect and extract extended-analysis ZIP
# ----------------------------------------
RESULT_ZIP_PATHS = []
if AUTO_EXTRACT_RESULTS_ZIP:
    RESULT_ZIP_PATHS = recursive_find(
        INPUT_ROOT,
        lambda p: p.is_file()
        and p.suffix.lower() == ".zip"
        and "molm_unified_experiment_results" in p.name.lower()
    )

print("\nDetected unified result ZIPs:")
if RESULT_ZIP_PATHS:
    for p in RESULT_ZIP_PATHS:
        print("  ", p)
else:
    print("   none")

RESULT_SEARCH_ROOTS = [INPUT_ROOT]

if RESULT_ZIP_PATHS:
    zip_path = RESULT_ZIP_PATHS[0]
    marker = EXISTING_RESULTS_DIR / ".extracted_from.json"
    current_sig = {
        "path": str(zip_path),
        "size": int(zip_path.stat().st_size),
        "mtime_ns": int(zip_path.stat().st_mtime_ns),
    }

    need_extract = True
    if marker.exists():
        try:
            old_sig = json.loads(marker.read_text())
            if old_sig == current_sig and any(EXISTING_RESULTS_DIR.iterdir()):
                need_extract = False
        except Exception:
            pass

    if need_extract:
        print("\nExtracting result ZIP...")
        for p in EXISTING_RESULTS_DIR.iterdir():
            if p.name == marker.name:
                continue
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(EXISTING_RESULTS_DIR)
        marker.write_text(json.dumps(current_sig, indent=2))
        print("Extracted to:", EXISTING_RESULTS_DIR)
    else:
        print("\nResult ZIP already extracted; reusing:", EXISTING_RESULTS_DIR)

    RESULT_SEARCH_ROOTS.insert(0, EXISTING_RESULTS_DIR)

# Quick inventory of useful existing tables.
wanted_names = {
    "pareto_fixed_budget_summary_4models.csv",
    "pareto_fixed_budget_by_run_4models.csv",
    "pareto_architecture_targeted_contrasts_all_k.csv",
    "MASTER_4MODEL_SUMMARY.csv",
    "RESULT_COVERAGE_MATRIX.csv",
}
print("\nUseful existing files found:")
found_any = False
for root in RESULT_SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.is_file() and p.name in wanted_names:
            found_any = True
            print("  ", p)
if not found_any:
    print("   none of the expected filenames were found yet")


## 2. Build the exact five representations


In [ ]:

import gc
from scipy.stats import spearmanr

sys.path.insert(0, str(REPO))
import phase0_config as pc

# Explicit scientific/training audit.
cfg = pc.config
assert list(cfg.SHARED_DIMS) == [256, 128]
assert list(cfg.TOWER_DIMS) == [64, 32]
assert int(cfg.LATENT_DIM) == 16
assert float(cfg.DROPOUT_RATE) == 0.2
assert float(cfg.RANKING_WEIGHT_AFF) == 0.3
assert float(cfg.RANKING_WEIGHT_SPEC) == 0.6
assert float(cfg.GAP_WEIGHT_AFF) == 0.2
assert float(cfg.GAP_WEIGHT_SPEC) == 0.6
assert float(cfg.RANKING_MARGIN) == 0.3
assert float(cfg.GAP_MARGIN) == 0.2
assert list(cfg.MUTATION_SITES) == [32, 49, 54, 55, 56, 98, 100, 103]

def resolve_column(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError(f"None of {candidates} found. Columns={list(frame.columns)}")

emi_binding = pd.read_csv(REPO / "data" / "emi_binding.csv", index_col=0)
iso_binding = pd.read_csv(REPO / "data" / "iso_binding.csv", index_col=0)
igg_binding = pd.read_csv(REPO / "data" / "igg_binding.csv", index_col=0)

emi_sequences = emi_binding.index.astype(str).to_numpy()
iso_sequences = iso_binding.index.astype(str).to_numpy()
igg_sequences = igg_binding.index.astype(str).to_numpy()

for name, seqs in [("EMI", emi_sequences), ("ISO", iso_sequences), ("IgG", igg_sequences)]:
    lengths = np.array([len(s) for s in seqs])
    if not np.all(lengths == 115):
        raise ValueError(f"{name}: expected aligned length 115, observed {sorted(set(lengths.tolist()))}")

emi_aff_col = resolve_column(emi_binding, ["ANT Binding", "ANT", "Affinity", "affinity"])
emi_ova_col = resolve_column(emi_binding, ["OVA Binding", "OVA", "PSY", "Specificity", "specificity"])
iso_aff_col = resolve_column(iso_binding, ["ANT Binding", "ANT", "Affinity", "affinity"])
iso_ova_col = resolve_column(iso_binding, ["OVA Binding", "OVA", "PSY", "Specificity", "specificity"])
igg_aff_col = resolve_column(igg_binding, ["ANT Binding", "ANT", "Affinity", "affinity"])
igg_ova_col = resolve_column(igg_binding, ["OVA Binding", "OVA", "PSY", "Specificity", "specificity"])

y_aff_emi = (emi_binding[emi_aff_col].to_numpy() > 0).astype(np.float32)
y_ova_emi = (emi_binding[emi_ova_col].to_numpy() > 0).astype(np.float32)

iso_aff = iso_binding[iso_aff_col].to_numpy(float)
iso_ova = iso_binding[iso_ova_col].to_numpy(float)
igg_aff = igg_binding[igg_aff_col].to_numpy(float)
igg_ova = igg_binding[igg_ova_col].to_numpy(float)

aff_pos_weight = float((1.0 - y_aff_emi.mean()) / max(float(y_aff_emi.mean()), 1e-6))
spec_pos_weight = 1.0  # exact phase1_features.py behavior

np.savez(
    FEATURE_DIR / "labels_and_truth.npz",
    y_aff_emi=y_aff_emi,
    y_ova_emi=y_ova_emi,
    iso_aff=iso_aff,
    iso_ova=iso_ova,
    igg_aff=igg_aff,
    igg_ova=igg_ova,
    aff_pos_weight=np.array([aff_pos_weight], dtype=np.float32),
    spec_pos_weight=np.array([spec_pos_weight], dtype=np.float32),
)

pd.DataFrame({"sequence_id": iso_sequences}).to_csv(FEATURE_DIR / "iso_sequence_ids.csv", index=False)
pd.DataFrame({"sequence_id": igg_sequences}).to_csv(FEATURE_DIR / "igg_sequence_ids.csv", index=False)

print("EMI:", len(emi_sequences), "ISO:", len(iso_sequences), "IgG:", len(igg_sequences))
print("Target positive fraction:", y_aff_emi.mean())
print("Target positive weight:", aff_pos_weight)
print("OVA positive fraction:", y_ova_emi.mean())


In [ ]:

# Build / reuse OneHot and ESM-derived arrays.
import gc
import esm

def feature_path(dataset, feature):
    return FEATURE_DIR / f"{dataset}_{feature}.npy"

DATASET_ROW_COUNTS = {
    "emi": len(emi_sequences),
    "iso": len(iso_sequences),
    "igg": len(igg_sequences),
}
EXPECTED_ESM_DIMS = {
    "mean_esm2": 320,
    "site_esm2": 2560,
}
SEQ_INDEX = {
    "emi": pd.Index(emi_sequences.astype(str)),
    "iso": pd.Index(iso_sequences.astype(str)),
    "igg": pd.Index(igg_sequences.astype(str)),
}

# ----------------------------------------------------------
# Helpers for safely reusing ESM arrays from a Kaggle dataset
# ----------------------------------------------------------
def _infer_dataset_from_name(path):
    s = path.stem.lower()
    for ds in ["emi", "iso", "igg"]:
        if re.search(rf"(^|[_\-.]){ds}([_\-.]|$)", s):
            return ds
    return None

def _infer_kind_from_name(path):
    s = path.stem.lower()
    if "site" in s and ("esm" in s or "embedding" in s):
        return "site_esm2"
    if ("mean" in s and ("esm" in s or "embedding" in s)) or re.search(r"(^|[_\-.])esm2?([_\-.]|$)", s):
        return "mean_esm2"
    return None

def _read_candidate_matrix(path, ds, kind):
    """
    Return float32 ndarray aligned to the repository sequence order, or None.
    CSV/TSV files with sequence IDs are aligned explicitly.
    NPY files are accepted only under strict shape verification.
    """
    n_expected = DATASET_ROW_COUNTS[ds]
    d_expected = EXPECTED_ESM_DIMS[kind]

    try:
        if path.suffix.lower() == ".npy":
            if not ALLOW_SHAPE_VERIFIED_NPY_REUSE:
                return None
            arr = np.load(path, mmap_mode="r")
            if arr.ndim != 2 or arr.shape != (n_expected, d_expected):
                return None
            return np.asarray(arr, dtype=np.float32)

        if path.suffix.lower() == ".npz":
            z = np.load(path)
            for key in z.files:
                arr = z[key]
                if arr.ndim == 2 and arr.shape == (n_expected, d_expected):
                    return np.asarray(arr, dtype=np.float32)
            return None

        if path.suffix.lower() in [".csv", ".tsv", ".txt"]:
            sep = "\t" if path.suffix.lower() in [".tsv", ".txt"] else ","

            # First try indexed CSV, which is what the original pipeline writes.
            df = pd.read_csv(path, sep=sep, index_col=0)
            numeric = df.select_dtypes(include=[np.number])

            # Best case: sequence IDs are in index, so align explicitly.
            idx_as_str = pd.Index(df.index.astype(str))
            target_idx = SEQ_INDEX[ds]
            if target_idx.isin(idx_as_str).all() and numeric.shape[1] >= d_expected:
                numeric.index = idx_as_str
                # Prefer exactly d_expected columns, otherwise first compatible numeric block.
                mat = numeric.loc[target_idx]
                if mat.shape[1] == d_expected:
                    return mat.to_numpy(np.float32)

                # Common case: sequence/id metadata plus ESM numeric columns.
                esm_cols = [
                    c for c in mat.columns
                    if ("esm" in str(c).lower()) or re.match(r"^\d+$", str(c))
                ]
                if len(esm_cols) == d_expected:
                    return mat[esm_cols].to_numpy(np.float32)

            # Fallback: shape-verified numeric matrix without usable IDs.
            if ALLOW_SHAPE_VERIFIED_NPY_REUSE:
                if numeric.shape == (n_expected, d_expected):
                    return numeric.to_numpy(np.float32)

                # Sometimes index_col=0 consumed a numeric embedding column.
                df2 = pd.read_csv(path, sep=sep)
                numeric2 = df2.select_dtypes(include=[np.number])
                if numeric2.shape == (n_expected, d_expected):
                    return numeric2.to_numpy(np.float32)

            return None
    except Exception:
        return None

def discover_reusable_esm():
    """
    Scan explicit esm_reuse dataset roots and infer compatible ESM matrices by:
      - dataset name hint (EMI/ISO/IgG)
      - representation hint (mean/site)
      - strict expected shape
    """
    accepted = {}
    if not ESM_REUSE_ROOTS:
        return accepted

    candidate_files = []
    for root in ESM_REUSE_ROOTS:
        for ext in ("*.npy", "*.npz", "*.csv", "*.tsv", "*.txt"):
            candidate_files.extend(root.rglob(ext))

    # Prefer filenames that explicitly identify dataset + representation.
    candidate_files = sorted(
        set(candidate_files),
        key=lambda p: (
            0 if _infer_dataset_from_name(p) else 1,
            0 if _infer_kind_from_name(p) else 1,
            len(str(p)),
        ),
    )

    for path in candidate_files:
        ds_hint = _infer_dataset_from_name(path)
        kind_hint = _infer_kind_from_name(path)

        ds_options = [ds_hint] if ds_hint else ["emi", "iso", "igg"]
        kind_options = [kind_hint] if kind_hint else ["mean_esm2", "site_esm2"]

        for ds in ds_options:
            for kind in kind_options:
                key = (ds, kind)
                if key in accepted:
                    continue
                mat = _read_candidate_matrix(path, ds, kind)
                if mat is not None:
                    accepted[key] = (path, mat)
                    break

    return accepted

# ----------------------------
# 1) Build exact repository OneHot
# ----------------------------
print("Building / checking OneHot...")
onehot = {
    "emi": pc.generate_onehot(emi_binding).values.astype(np.float32),
    "iso": pc.generate_onehot(iso_binding).values.astype(np.float32),
    "igg": pc.generate_onehot(igg_binding).values.astype(np.float32),
}
for ds, arr in onehot.items():
    assert arr.shape == (DATASET_ROW_COUNTS[ds], 2300)
    np.save(feature_path(ds, "onehot"), arr)
print("  OneHot ready.")

# -----------------------------------------
# 2) Reuse compatible ESM arrays if present
# -----------------------------------------
reusable = discover_reusable_esm() if AUTO_DETECT_ESM_REUSE else {}

print("\nESM cache reuse audit:")
for ds in ["emi", "iso", "igg"]:
    for kind in ["mean_esm2", "site_esm2"]:
        if kind == "site_esm2" and not RUN_SITE_FEATURES:
            continue
        key = (ds, kind)
        dst = feature_path(ds, kind)

        if RESUME and dst.exists():
            arr = np.load(dst, mmap_mode="r")
            if arr.shape == (DATASET_ROW_COUNTS[ds], EXPECTED_ESM_DIMS[kind]):
                print(f"  WORKING CACHE  {ds:3s} {kind:10s} <- {dst}")
                continue

        if key in reusable:
            src_path, mat = reusable[key]
            assert mat.shape == (DATASET_ROW_COUNTS[ds], EXPECTED_ESM_DIMS[kind])
            np.save(dst, mat.astype(np.float32, copy=False))
            print(f"  REUSED         {ds:3s} {kind:10s} <- {src_path}")
        else:
            print(f"  MISSING        {ds:3s} {kind:10s} -> will compute if needed")

# --------------------------------------------
# 3) Compute only ESM representations missing
# --------------------------------------------
requested_esm_kinds = ["mean_esm2"] + (["site_esm2"] if RUN_SITE_FEATURES else [])

missing_pairs = [
    (ds, kind)
    for ds in ["emi", "iso", "igg"]
    for kind in requested_esm_kinds
    if not feature_path(ds, kind).exists()
]

if missing_pairs:
    print("\nMissing ESM feature pairs:", missing_pairs)
    print("Loading ESM-2 esm2_t6_8M_UR50D only for missing features...")

    esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    esm_model.eval()
    device = torch.device("cuda:0")
    esm_model = esm_model.to(device)
    batch_converter = alphabet.get_batch_converter()

    site_seq_indices = list(map(int, cfg.MUTATION_SITES))
    site_token_indices = [i + 1 for i in site_seq_indices]  # BOS is token 0

    @torch.no_grad()
    def compute_needed(sequences, need_mean, need_site, batch_size=64):
        means = [] if need_mean else None
        sites = [] if need_site else None
        tuples = [(f"s{i}", str(s)) for i, s in enumerate(sequences)]

        for start in range(0, len(tuples), batch_size):
            batch = tuples[start:start+batch_size]
            _, _, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            out = esm_model(tokens, repr_layers=[6], return_contacts=False)
            reps = out["representations"][6]

            for j, (_, seq) in enumerate(batch):
                L = len(seq)
                if need_mean:
                    residue_reps = reps[j, 1:L+1, :]
                    means.append(residue_reps.mean(dim=0).float().cpu().numpy())
                if need_site:
                    sites.append(
                        reps[j, site_token_indices, :].reshape(-1).float().cpu().numpy()
                    )

            if start == 0 or (start // batch_size + 1) % 10 == 0:
                print(f"  {min(start+batch_size, len(tuples))}/{len(tuples)}")

        mean_arr = np.stack(means).astype(np.float32) if need_mean else None
        site_arr = np.stack(sites).astype(np.float32) if need_site else None
        return mean_arr, site_arr

    for ds, seqs in [("emi", emi_sequences), ("iso", iso_sequences), ("igg", igg_sequences)]:
        need_mean = not feature_path(ds, "mean_esm2").exists()
        need_site = RUN_SITE_FEATURES and not feature_path(ds, "site_esm2").exists()

        if not (need_mean or need_site):
            continue

        print(f"\nComputing missing {ds.upper()} ESM features: mean={need_mean}, site={need_site}")
        mean_arr, site_arr = compute_needed(seqs, need_mean, need_site, batch_size=64)

        if need_mean:
            assert mean_arr.shape == (len(seqs), 320)
            np.save(feature_path(ds, "mean_esm2"), mean_arr)
        if need_site:
            assert site_arr.shape == (len(seqs), 2560)
            np.save(feature_path(ds, "site_esm2"), site_arr)

    del esm_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("\nAll requested ESM features were reused/cached. No ESM model inference needed.")

# --------------------------
# 4) Build fusion features
# --------------------------
for ds in ["emi", "iso", "igg"]:
    oh = np.load(feature_path(ds, "onehot"))
    mean_e = np.load(feature_path(ds, "mean_esm2"))
    mean_f = np.concatenate([oh, mean_e], axis=1).astype(np.float32)
    assert mean_f.shape == (DATASET_ROW_COUNTS[ds], 2620)
    np.save(feature_path(ds, "mean_fusion"), mean_f)

    if RUN_SITE_FEATURES:
        site_e = np.load(feature_path(ds, "site_esm2"))
        site_f = np.concatenate([oh, site_e], axis=1).astype(np.float32)
        assert site_f.shape == (DATASET_ROW_COUNTS[ds], 4860)
        np.save(feature_path(ds, "site_fusion"), site_f)

print("\nFinal feature audit:")
for f in FEATURE_TYPES:
    shapes = {
        ds: tuple(np.load(feature_path(ds, f), mmap_mode="r").shape)
        for ds in ["emi", "iso", "igg"]
    }
    expected_dim = FEATURE_DIMS[f]
    assert all(s[1] == expected_dim for s in shapes.values())
    print(f"  {f:12s} {shapes}")



## 3. Exact MOLM-ST worker

Each worker imports the **original** `train_molm_st()` implementation from the pinned repository.

For a given seed and representation it trains:

1. a target-binding MOLM-ST model with original seed offset `+500`
2. an OVA-binding MOLM-ST model with original seed offset `+600`

The architecture itself is unchanged; the distinction from Standard-MOLM is that the two models are trained independently and only the selected task loss updates each copy.


In [ ]:
WORKER_PATH = CODE_DIR / 'molm_st_fixed_budget_worker.py'
WORKER_SOURCE = '\nfrom __future__ import annotations\nimport os, sys, gc, json, time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.stats import spearmanr\n\nrepo = Path(os.environ["MOLM_REPO"])\nfeature_dir = Path(os.environ["MOLM_FEATURE_DIR"])\noutput_dir = Path(os.environ["MOLM_JOB_OUTPUT"])\nfeature = os.environ["MOLM_FEATURE"]\nseed = int(os.environ["MOLM_SEED"])\n\noutput_dir.mkdir(parents=True, exist_ok=True)\nsys.path.insert(0, str(repo))\n\n# IMPORTANT: MOLM_SEED is already in the environment before this import.\nimport phase0_config as pc\n\ncfg = pc.config\ncfg.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\ncfg.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\ncfg.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\ncfg.PARETO_LOSS = False\ncfg.ADVERSARIAL_WEIGHT = 0.0\ncfg.ORTHO_WEIGHT = 0.0\n\ndef fp(ds):\n    return feature_dir / f"{ds}_{feature}.npy"\n\nX_emi = np.load(fp("emi")).astype(np.float32, copy=False)\nX_iso = np.load(fp("iso")).astype(np.float32, copy=False)\nX_igg = np.load(fp("igg")).astype(np.float32, copy=False)\n\nlab = np.load(feature_dir / "labels_and_truth.npz")\ny_aff = lab["y_aff_emi"].astype(np.float32)\ny_ova = lab["y_ova_emi"].astype(np.float32)\niso_aff = lab["iso_aff"].astype(float)\niso_ova = lab["iso_ova"].astype(float)\nigg_aff = lab["igg_aff"].astype(float)\nigg_ova = lab["igg_ova"].astype(float)\naff_pw = float(lab["aff_pos_weight"][0])\nspec_pw = float(lab["spec_pos_weight"][0])\n\niso_ids = pd.read_csv(feature_dir / "iso_sequence_ids.csv")["sequence_id"].astype(str).to_numpy()\nigg_ids = pd.read_csv(feature_dir / "igg_sequence_ids.csv")["sequence_id"].astype(str).to_numpy()\n\ndef predict_task(model, X, task):\n    model.eval()\n    device = next(model.parameters()).device\n    with torch.no_grad():\n        out = model(torch.as_tensor(X, dtype=torch.float32, device=device), training=False)\n    key = "aff_score" if task == "affinity" else "spec_score"\n    return out[key].detach().float().cpu().numpy().reshape(-1)\n\nt0 = time.time()\n\n# Original MOLM-ST affinity model.\nm_aff = pc.train_molm_st(\n    X_emi, y_aff, y_ova,\n    "affinity", cfg,\n    aff_pos_weight=aff_pw,\n    spec_pos_weight=spec_pw,\n    seed_offset=500,\n    verbose=0,\n)\npred_aff_iso = predict_task(m_aff, X_iso, "affinity")\npred_aff_igg = predict_task(m_aff, X_igg, "affinity")\nn_params_single = int(sum(p.numel() for p in m_aff.parameters()))\ndel m_aff\ngc.collect()\ntorch.cuda.empty_cache()\n\n# Original MOLM-ST specificity / OVA model.\nm_ova = pc.train_molm_st(\n    X_emi, y_aff, y_ova,\n    "specificity", cfg,\n    aff_pos_weight=aff_pw,\n    spec_pos_weight=spec_pw,\n    seed_offset=600,\n    verbose=0,\n)\npred_ova_iso = predict_task(m_ova, X_iso, "specificity")\npred_ova_igg = predict_task(m_ova, X_igg, "specificity")\ndel m_ova\ngc.collect()\ntorch.cuda.empty_cache()\n\nframes = []\n\ndef add_dataset(name, ids, pa, po, ta, to):\n    frames.append(pd.DataFrame({\n        "seed": seed,\n        "feature": feature,\n        "dataset": name,\n        "model": "MOLM-ST",\n        "row_id": np.arange(len(ids), dtype=int),\n        "sequence_id": np.asarray(ids).astype(str),\n        "pred_aff": np.asarray(pa, float),\n        "pred_ova": np.asarray(po, float),\n        "true_aff": np.asarray(ta, float),\n        "true_ova": np.asarray(to, float),\n    }))\n\nadd_dataset("ISO", iso_ids, pred_aff_iso, pred_ova_iso, iso_aff, iso_ova)\nadd_dataset("IgG-primary42", igg_ids[:42], pred_aff_igg[:42], pred_ova_igg[:42], igg_aff[:42], igg_ova[:42])\nadd_dataset("IgG-all96", igg_ids, pred_aff_igg, pred_ova_igg, igg_aff, igg_ova)\n\npred = pd.concat(frames, ignore_index=True)\npred.to_csv(output_dir / "external_predictions.csv.gz", index=False, compression="gzip")\n\nspearman_rows = []\nfor dataset, g in pred.groupby("dataset", sort=False):\n    ar = float(spearmanr(g["pred_aff"], g["true_aff"]).statistic)\n    orho = float(spearmanr(g["pred_ova"], g["true_ova"]).statistic)\n    spearman_rows.append({\n        "seed": seed,\n        "feature": feature,\n        "dataset": dataset,\n        "model": "MOLM-ST",\n        "aff_spearman": ar,\n        "ova_spearman": orho,\n        "n": len(g),\n    })\npd.DataFrame(spearman_rows).to_csv(output_dir / "external_spearman.csv", index=False)\n\nmeta = {\n    "seed": seed,\n    "feature": feature,\n    "repository_commit": os.environ.get("MOLM_COMMIT"),\n    "epochs": cfg.EPOCHS,\n    "batch_size": cfg.BATCH_SIZE,\n    "learning_rate": cfg.LEARNING_RATE,\n    "single_model_parameter_count": n_params_single,\n    "two_independent_model_parameter_count": 2 * n_params_single,\n    "runtime_seconds": time.time() - t0,\n    "gpu_visible": os.environ.get("CUDA_VISIBLE_DEVICES"),\n}\n(output_dir / "JOB_DONE.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")\nprint(json.dumps(meta, indent=2))\n'
WORKER_PATH.write_text(WORKER_SOURCE, encoding='utf-8')
print('Worker written:', WORKER_PATH)



## 4. Run all 25 seed × representation jobs on T4×2

The scheduler uses one sequential queue per GPU, so two training jobs run at once without accidentally placing two workers on the same T4.

Completed jobs are resumable.


In [ ]:

from concurrent.futures import ThreadPoolExecutor

def job_output(seed, feature):
    return JOBS_DIR / f"seed_{seed}" / feature

def job_complete(seed, feature):
    root = job_output(seed, feature)
    return (root / "JOB_DONE.json").exists() and (root / "external_predictions.csv.gz").exists()

def run_one_job(seed, feature, gpu):
    root = job_output(seed, feature)
    root.mkdir(parents=True, exist_ok=True)

    if RESUME and job_complete(seed, feature):
        meta = json.loads((root / "JOB_DONE.json").read_text())
        print(f"[GPU {gpu}] SKIP seed={seed} feature={feature} ({meta.get('runtime_seconds', 0)/60:.1f} min cached)")
        return {"seed": seed, "feature": feature, "gpu": gpu, "status": "skipped"}

    log_path = LOG_DIR / f"seed_{seed}_{feature}.log"
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "MOLM_REPO": str(REPO),
        "MOLM_FEATURE_DIR": str(FEATURE_DIR),
        "MOLM_JOB_OUTPUT": str(root),
        "MOLM_FEATURE": feature,
        "MOLM_SEED": str(seed),
        "MOLM_EPOCHS": str(EPOCHS),
        "MOLM_BATCH_SIZE": str(BATCH_SIZE),
        "MOLM_LEARNING_RATE": str(LEARNING_RATE),
        "MOLM_COMMIT": PINNED_COMMIT,
        "PYTHONPATH": str(REPO),
    })

    print(f"[GPU {gpu}] START seed={seed} feature={feature}")
    t0 = time.time()
    with log_path.open("w") as log:
        p = subprocess.run(
            [sys.executable, "-u", str(WORKER_PATH)],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

    if p.returncode != 0:
        tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-120:])
        raise RuntimeError(f"Job failed seed={seed}, feature={feature}, gpu={gpu}\n{tail}")

    elapsed = time.time() - t0
    print(f"[GPU {gpu}] DONE  seed={seed} feature={feature} in {elapsed/60:.1f} min")
    return {"seed": seed, "feature": feature, "gpu": gpu, "status": "ok", "runtime_seconds": elapsed}

# Greedy load-balance by input dimension; large representations are scheduled first.
jobs = [(seed, feature) for seed in SEEDS for feature in FEATURE_TYPES]
jobs = sorted(jobs, key=lambda x: FEATURE_DIMS[x[1]], reverse=True)

queues = {0: [], 1: []}
loads = {0: 0, 1: 0}
for job in jobs:
    gpu = min(loads, key=loads.get)
    queues[gpu].append(job)
    loads[gpu] += FEATURE_DIMS[job[1]]

print("Approximate queue loads:", loads)
print("GPU0 jobs:", queues[0])
print("GPU1 jobs:", queues[1])

def run_gpu_queue(gpu):
    out = []
    for seed, feature in queues[gpu]:
        out.append(run_one_job(seed, feature, gpu))
    return out

all_t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as ex:
    fut0 = ex.submit(run_gpu_queue, 0)
    fut1 = ex.submit(run_gpu_queue, 1)
    results = fut0.result() + fut1.result()

manifest = pd.DataFrame(results)
manifest.to_csv(ANALYSIS_DIR / "molm_st_job_manifest.csv", index=False)
display(manifest)
print(f"Training wall time: {(time.time()-all_t0)/60:.1f} min")



## 5. Collect predictions and external Spearman results


In [ ]:

pred_frames = []
sp_frames = []
meta_rows = []

for seed in SEEDS:
    for feature in FEATURE_TYPES:
        root = job_output(seed, feature)
        if not job_complete(seed, feature):
            raise RuntimeError(f"Missing completed job: seed={seed}, feature={feature}")
        pred_frames.append(pd.read_csv(root / "external_predictions.csv.gz"))
        sp_frames.append(pd.read_csv(root / "external_spearman.csv"))
        meta_rows.append(json.loads((root / "JOB_DONE.json").read_text()))

external_raw = pd.concat(pred_frames, ignore_index=True)
spearman_by_seed = pd.concat(sp_frames, ignore_index=True)
job_meta = pd.DataFrame(meta_rows)

external_raw.to_csv(
    ANALYSIS_DIR / "molm_st_external_predictions_all.csv.gz",
    index=False,
    compression="gzip",
)
spearman_by_seed.to_csv(
    ANALYSIS_DIR / "molm_st_external_spearman_by_seed.csv",
    index=False,
)

spearman_summary = (
    spearman_by_seed
    .groupby(["dataset", "feature", "model"], as_index=False)
    .agg(
        n_seeds=("seed", "size"),
        aff_spearman_mean=("aff_spearman", "mean"),
        aff_spearman_sd=("aff_spearman", "std"),
        ova_spearman_mean=("ova_spearman", "mean"),
        ova_spearman_sd=("ova_spearman", "std"),
    )
)
spearman_summary.to_csv(
    ANALYSIS_DIR / "molm_st_external_spearman_summary.csv",
    index=False,
)

display(spearman_summary[spearman_summary.dataset == "ISO"])
print("Median completed job runtime:", f"{job_meta.runtime_seconds.median()/60:.1f} min")



## 6. Fixed-budget Pareto evaluation

This uses the same deterministic fixed-budget selection rule as the extended-analysis notebook:

1. non-dominated sorting in predicted `(target, -OVA)` score space;
2. take complete fronts until the next front would exceed the budget;
3. resolve the final partial front with crowding distance;
4. deterministic sequence-ID tie breaking.

HV and IGD are calculated in min-max normalized **measured** objective space.


In [ ]:

def pareto_mask_max(points):
    points = np.asarray(points, float)
    mask = np.ones(len(points), bool)
    for i in range(len(points)):
        if (np.all(points >= points[i], axis=1) & np.any(points > points[i], axis=1)).any():
            mask[i] = False
    return mask

def nondominated_sort(points):
    remaining = np.arange(len(points))
    fronts = []
    while len(remaining):
        mask = pareto_mask_max(points[remaining])
        fronts.append(remaining[mask])
        remaining = remaining[~mask]
    return fronts

def crowding_distance(points):
    points = np.asarray(points, float)
    d = np.zeros(len(points), float)
    if len(points) <= 2:
        d[:] = np.inf
        return d
    for obj in range(points.shape[1]):
        order = np.argsort(points[:, obj], kind="mergesort")
        d[order[0]] = d[order[-1]] = np.inf
        span = points[order[-1], obj] - points[order[0], obj]
        if span <= 0:
            continue
        for r in range(1, len(points) - 1):
            cur = order[r]
            if np.isfinite(d[cur]):
                d[cur] += (
                    points[order[r+1], obj] - points[order[r-1], obj]
                ) / span
    return d

def select_fixed_budget(points, identifiers, k):
    points = np.asarray(points, float)
    ids = np.asarray(identifiers).astype(str)
    selected = []
    for front in nondominated_sort(points):
        if len(selected) + len(front) <= k:
            selected.extend(front.tolist())
            continue
        rem = k - len(selected)
        dist = crowding_distance(points[front])
        order = sorted(
            range(len(front)),
            key=lambda j: (-dist[j], ids[front[j]])
        )
        selected.extend(front[order[:rem]].tolist())
        break
    return np.asarray(selected, int)

def normalize_true_objectives(a, o):
    p = np.c_[np.asarray(a, float), -np.asarray(o, float)]
    mn = p.min(axis=0)
    sp = p.max(axis=0) - mn
    sp[sp == 0] = 1.0
    return (p - mn) / sp

def hypervolume_2d_max(points):
    points = np.asarray(points, float)
    points = points[np.all(points >= 0, axis=1)]
    if not len(points):
        return 0.0
    points = points[pareto_mask_max(points)]
    points = points[np.argsort(points[:, 0])]
    total = 0.0
    prev = 0.0
    for x, y in points:
        total += max(0.0, x - prev) * max(0.0, y)
        prev = max(prev, x)
    return float(total)

def igd(true_front, selected):
    return float(
        np.sqrt(
            ((true_front[:, None, :] - selected[None, :, :]) ** 2).sum(axis=2)
        ).min(axis=1).mean()
    )

def recall_auc(g):
    g = g.sort_values("k")
    x = g.k.to_numpy(float)
    y = g.recall_at_k.to_numpy(float)
    return float(np.trapz(y, x) / (x[-1] - x[0])) if len(x) > 1 else np.nan

budget_rows = []
curve_rows = []

for (seed, feature, dataset, model), g in external_raw.groupby(
    ["seed", "feature", "dataset", "model"], sort=False
):
    g = g.sort_values("row_id").reset_index(drop=True)

    pred_points = np.c_[
        g.pred_aff.to_numpy(float),
        -g.pred_ova.to_numpy(float),
    ]
    measured_points_raw = np.c_[
        g.true_aff.to_numpy(float),
        -g.true_ova.to_numpy(float),
    ]
    true_mask = pareto_mask_max(measured_points_raw)
    true_norm = normalize_true_objectives(g.true_aff, g.true_ova)

    n_true = int(true_mask.sum())
    prevalence = n_true / len(g)

    for k in range(1, min(FULL_CURVE_MAX_K, len(g)) + 1):
        sel = select_fixed_budget(pred_points, g.sequence_id, k)
        hits = int(true_mask[sel].sum())

        curve_rows.append({
            "seed": int(seed),
            "feature": feature,
            "dataset": dataset,
            "model": model,
            "k": k,
            "hits": hits,
            "recall_at_k": hits / max(n_true, 1),
        })

        if k in K_VALUES:
            precision = hits / k
            budget_rows.append({
                "seed": int(seed),
                "feature": feature,
                "dataset": dataset,
                "model": model,
                "k": k,
                "true_front_n": n_true,
                "hits": hits,
                "recall_at_k": hits / max(n_true, 1),
                "precision_at_k": precision,
                "enrichment_at_k": precision / prevalence,
                "hypervolume_true_selected": hypervolume_2d_max(true_norm[sel]),
                "igd_true_front_to_selected": igd(true_norm[true_mask], true_norm[sel]),
            })

pareto_by_run = pd.DataFrame(budget_rows)
pareto_curve = pd.DataFrame(curve_rows)

pareto_summary = (
    pareto_by_run
    .groupby(["dataset", "feature", "model", "k"], as_index=False)
    .agg(
        n_runs=("seed", "size"),
        hits_mean=("hits", "mean"),
        hits_sd=("hits", "std"),
        recall_mean=("recall_at_k", "mean"),
        recall_sd=("recall_at_k", "std"),
        precision_mean=("precision_at_k", "mean"),
        precision_sd=("precision_at_k", "std"),
        enrichment_mean=("enrichment_at_k", "mean"),
        enrichment_sd=("enrichment_at_k", "std"),
        hypervolume_mean=("hypervolume_true_selected", "mean"),
        hypervolume_sd=("hypervolume_true_selected", "std"),
        igd_mean=("igd_true_front_to_selected", "mean"),
        igd_sd=("igd_true_front_to_selected", "std"),
    )
)

auc_by_run = pd.DataFrame([
    {
        "seed": int(seed),
        "feature": feature,
        "dataset": dataset,
        "model": model,
        "recall_auc_k1_25": recall_auc(g),
    }
    for (seed, feature, dataset, model), g in pareto_curve.groupby(
        ["seed", "feature", "dataset", "model"]
    )
])

auc_summary = (
    auc_by_run
    .groupby(["dataset", "feature", "model"], as_index=False)
    .agg(
        auc_mean=("recall_auc_k1_25", "mean"),
        auc_sd=("recall_auc_k1_25", "std"),
    )
)

pareto_by_run.to_csv(ANALYSIS_DIR / "molm_st_pareto_fixed_budget_by_run.csv", index=False)
pareto_curve.to_csv(ANALYSIS_DIR / "molm_st_pareto_recall_curve_k1_25.csv", index=False)
pareto_summary.to_csv(ANALYSIS_DIR / "molm_st_pareto_fixed_budget_summary.csv", index=False)
auc_by_run.to_csv(ANALYSIS_DIR / "molm_st_pareto_recall_auc_by_run.csv", index=False)
auc_summary.to_csv(ANALYSIS_DIR / "molm_st_pareto_recall_auc_summary.csv", index=False)

display(
    pareto_summary[
        (pareto_summary.dataset == "ISO") &
        (pareto_summary.k.isin(K_VALUES))
    ].sort_values(["feature", "k"])
)


## 7. Optional merge with existing four-model results

If a compatible result dataset or ZIP is attached, this section discovers the prior four-model fixed-budget tables and constructs merged five-model summaries. The MOLM-ST training/evaluation itself remains independent of this optional merge.


In [ ]:

def find_input_file(names):
    """
    Search extracted unified existing results first, then raw /kaggle/input.
    This lets the notebook use MOLM_Unified_Experiment_Results.zip
    automatically when the user mounts that ZIP as a Kaggle dataset.
    """
    if isinstance(names, str):
        names = [names]

    search_roots = []
    if "RESULT_SEARCH_ROOTS" in globals():
        search_roots.extend(RESULT_SEARCH_ROOTS)
    else:
        search_roots.extend([EXISTING_RESULTS_DIR, INPUT_ROOT])

    seen = set()
    ordered_roots = []
    for r in search_roots:
        r = Path(r)
        if str(r) not in seen and r.exists():
            seen.add(str(r))
            ordered_roots.append(r)

    for name in names:
        for root in ordered_roots:
            hits = sorted(root.rglob(name))
            if hits:
                return hits[0]
    return None

base_summary_path = find_input_file([
    "pareto_fixed_budget_summary_4models.csv",
])
base_run_path = find_input_file([
    "pareto_fixed_budget_by_run_4models.csv",
])

five_model_summary = None
contrast_table = None

if base_summary_path is not None:
    base_summary = pd.read_csv(base_summary_path)

    # Harmonize columns: retain the common summary columns.
    st = pareto_summary.copy()
    common = [c for c in base_summary.columns if c in st.columns]
    five_model_summary = pd.concat(
        [base_summary[common], st[common]],
        ignore_index=True,
    )
    five_model_summary.to_csv(
        ANALYSIS_DIR / "pareto_fixed_budget_summary_5models.csv",
        index=False,
    )
    print("Merged five-model summary:", base_summary_path)
else:
    print("No four-model summary found in /kaggle/input; skipping five-model merge.")

def exact_sign_flip_pvalue(diffs):
    d = np.asarray(diffs, float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return np.nan
    if np.allclose(d, 0):
        return 1.0
    observed = abs(float(d.mean()))
    vals = []
    for mask in range(1 << len(d)):
        signs = np.array([1.0 if (mask >> i) & 1 else -1.0 for i in range(len(d))])
        vals.append(abs(float((d * signs).mean())))
    vals = np.asarray(vals)
    return float((vals >= observed - 1e-15).mean())

def holm_adjust(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    out = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        adj = min(1.0, (m-rank) * p[idx])
        running = max(running, adj)
        out[idx] = running
    return out

if base_run_path is not None:
    base_run = pd.read_csv(base_run_path)
    combined_run = pd.concat([base_run, pareto_by_run], ignore_index=True)

    metric_specs = [
        ("recall_at_k", "higher"),
        ("precision_at_k", "higher"),
        ("enrichment_at_k", "higher"),
        ("hypervolume_true_selected", "higher"),
        ("igd_true_front_to_selected", "lower"),
    ]
    comparators = ["Routed-MOLM", "Standard-MOLM", "NN"]

    rows = []
    for dataset in sorted(set(pareto_by_run.dataset)):
        for feature in FEATURE_TYPES:
            for k in K_VALUES:
                block = combined_run[
                    (combined_run.dataset == dataset) &
                    (combined_run.feature == feature) &
                    (combined_run.k == k) &
                    (combined_run.seed.isin(SEEDS))
                ]
                for comp in comparators:
                    for metric, direction in metric_specs:
                        piv = block[block.model.isin([comp, "MOLM-ST"])].pivot_table(
                            index="seed", columns="model", values=metric, aggfunc="first"
                        )
                        if comp not in piv.columns or "MOLM-ST" not in piv.columns:
                            continue
                        piv = piv.dropna()
                        if len(piv) != len(SEEDS):
                            continue
                        raw_diff = piv[comp].to_numpy(float) - piv["MOLM-ST"].to_numpy(float)
                        oriented = raw_diff if direction == "higher" else -raw_diff
                        rows.append({
                            "dataset": dataset,
                            "feature": feature,
                            "k": k,
                            "comparison": f"{comp} - MOLM-ST",
                            "metric": metric,
                            "direction": direction,
                            "n_seeds": len(oriented),
                            "raw_mean_left_minus_molm_st": float(raw_diff.mean()),
                            "oriented_mean_advantage_left": float(oriented.mean()),
                            "wins_left": int((oriented > 0).sum()),
                            "ties": int(np.isclose(oriented, 0).sum()),
                            "losses_left": int((oriented < 0).sum()),
                            "exact_sign_flip_p": exact_sign_flip_pvalue(oriented),
                        })

    contrast_table = pd.DataFrame(rows)
    if len(contrast_table):
        # Holm within each dataset/feature/comparator/metric across the five K values.
        contrast_table["exact_sign_flip_p_holm_k5"] = np.nan
        for keys, idx in contrast_table.groupby(
            ["dataset", "feature", "comparison", "metric"]
        ).groups.items():
            idx = list(idx)
            contrast_table.loc[idx, "exact_sign_flip_p_holm_k5"] = holm_adjust(
                contrast_table.loc[idx, "exact_sign_flip_p"].to_numpy(float)
            )
        contrast_table.to_csv(
            ANALYSIS_DIR / "pareto_5model_molm_st_targeted_contrasts.csv",
            index=False,
        )

        display(
            contrast_table[
                (contrast_table.dataset == "ISO") &
                (contrast_table.comparison.isin([
                    "Routed-MOLM - MOLM-ST",
                    "Standard-MOLM - MOLM-ST",
                ]))
            ].sort_values(["feature", "comparison", "metric", "k"])
        )
else:
    print("No four-model by-run table found in /kaggle/input; paired five-model contrasts skipped.")



## 8. Final result bundle


In [ ]:

from datetime import datetime, timezone
import zipfile

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPO_URL,
    "repository_commit": PINNED_COMMIT,
    "model": "original MOLM-ST",
    "implementation": "phase0_config.train_molm_st",
    "seeds": SEEDS,
    "features": {f: FEATURE_DIMS[f] for f in FEATURE_TYPES},
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "architecture_shared_dims": [256, 128],
        "tower_dims": [64, 32],
        "latent_dim": 16,
        "dropout": 0.2,
        "target_ranking_weight": 0.3,
        "ova_ranking_weight": 0.6,
        "target_gap_weight": 0.2,
        "ova_gap_weight": 0.6,
        "ranking_margin": 0.3,
        "gap_margin": 0.2,
        "target_seed_offset": 500,
        "ova_seed_offset": 600,
    },
    "pareto": {
        "budgets": K_VALUES,
        "curve_max_k": FULL_CURVE_MAX_K,
        "orientation": "maximize target binding; minimize OVA binding",
        "selection": "non-dominated sorting + crowding distance + deterministic sequence-ID tie-break",
    },
}
(ANALYSIS_DIR / "MOLM_ST_REPRODUCIBILITY_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

# Include analysis tables, manifest, and exact worker.
bundle_dir = WORK_ROOT / "bundle"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir()

for p in ANALYSIS_DIR.iterdir():
    if p.is_file():
        shutil.copy2(p, bundle_dir / p.name)
shutil.copy2(WORKER_PATH, bundle_dir / WORKER_PATH.name)

zip_base = Path("/kaggle/working/MOLM_ST_FixedBudget_Pareto_Results")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=bundle_dir)

print("RESULT ZIP:", zip_path)
print("\nAnalysis files:")
for p in sorted(ANALYSIS_DIR.iterdir()):
    if p.is_file():
        print(" ", p.name, f"({p.stat().st_size/1024:.1f} KB)")



## Expected runtime on Kaggle T4×2

Because this version reuses `esm_reuse` automatically, ESM feature extraction may drop to essentially **0 minutes** if all six required Mean/Site matrices are found and verified.

For the complete five-representation run, a practical expectation is:

- setup + ZIP extraction: ~1–3 min
- ESM reuse verification: usually <1 min
- ESM recomputation: only for missing arrays
- MOLM-ST training on T4×2: roughly **25–45 min**
- Pareto analysis + merge + ZIP: <2 min

So with a complete compatible `esm_reuse` dataset, total wall time should usually be about **28–48 minutes**.

At the end upload back:

`MOLM_ST_FixedBudget_Pareto_Results.zip`

If the unified result ZIP contains the expected four-model tables, the result bundle will also contain merged five-model tables and targeted contrasts.
